# Mistral

In [1]:
##Removing annoying warnings
import warnings

#IProgress
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=".*IProgress not found.*"
)

#Suppress the torch/cuda AMD SMI warning
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=".*Can't initialize amdsmi.*"
)
warnings.filterwarnings("ignore")

In [2]:
import os
import torch
import gc

from datasets import load_dataset
from lightning import Trainer
from lightning.pytorch import LightningDataModule, LightningModule
from lightning.pytorch.callbacks import TQDMProgressBar
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel

os.environ["TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [3]:
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

LORA_DIR = ".ipynb_checkpoints/mistral7b_lora"

os.makedirs(".ipynb_checkpoints", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


# Data

In [4]:
def load_base_model():
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
    )

    return model

def attach_lora(model):
    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        lora_dropout=0.1,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

class DataModule(LightningDataModule):
    def __init__(self, batch_size=2, max_length=128):
        super().__init__()
        self.batch_size = batch_size
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def setup(self, stage=None):
        dataset = load_dataset(
            "Despina/project_gutenberg",
            "fiction_books",
            split="train",
            streaming=True
        ).shuffle(seed=42, buffer_size=1000)

        self.train_dataset = dataset.take(500)

    def collate_fn(self, batch):
        texts = [x["text"] for x in batch]

        enc = self.tokenizer(
            texts,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return enc["input_ids"], enc["input_ids"], enc["attention_mask"]

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            collate_fn=self.collate_fn
        )
    
class QLoRAModule(LightningModule):
    def __init__(self):
        super().__init__()
        base = load_base_model()

        for param in base.parameters():
            param.requires_grad = False

        self.model = attach_lora(base)

    def forward(self, input_ids, labels=None, attention_mask=None):
        return self.model(
            input_ids=input_ids,
            labels=labels,
            attention_mask=attention_mask
        )

    def training_step(self, batch, batch_idx):
        input_ids, labels, attention_mask = batch
        outputs = self(input_ids, labels, attention_mask)
        loss = outputs.loss
        self.log("train_loss", loss)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=2e-4)

# Finetuner

In [5]:
def train_model(epochs=1, batch_size=2):

    data = DataModule(batch_size=batch_size)
    model = QLoRAModule()

    progress_bar = TQDMProgressBar(refresh_rate=20)

    trainer = Trainer(
        max_epochs=epochs,
        accelerator="auto",
        logger=False,
        callbacks=[progress_bar]
    )

    trainer.fit(model, datamodule=data)

    #SAVE ONLY LORA ADAPTER
    model.model.save_pretrained(LORA_DIR)

    print("LoRA adapter saved to:", LORA_DIR)

    del model
    gc.collect()
    torch.cuda.empty_cache()

In [6]:
#train_model(epochs=1, batch_size=2)

# Generation

In [7]:
#Load inference and check if model is already in memory
#This way we don't have to realod the weights 15 million times
def setup_inference():
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    base_model = load_base_model()

    if os.path.exists(LORA_DIR):
        print("Loading LoRA adapter")
        model = PeftModel.from_pretrained(base_model, LORA_DIR)
    else:
        print("No LoRA adapter found. Using base model")
        model = base_model

    model.eval()
    return model, tokenizer

#Preload once
print("Loading model")
inference_model, inference_tokenizer = setup_inference()


def generate(prompt, max_length=150):
    formatted = f"<s>[INST] {prompt} [/INST]"

    inputs = inference_tokenizer(formatted, return_tensors="pt").to(inference_model.device)

    output = inference_model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        repetition_penalty=1.2,
        pad_token_id=inference_tokenizer.eos_token_id,
    )

    return inference_tokenizer.decode(output[0], skip_special_tokens=True)


Loading model


Loading weights: 100%|██████████| 291/291 [00:22<00:00, 12.68it/s, Materializing param=model.norm.weight]                               


Loading LoRA adapter


### Generate prompts

In [ ]:
OUTPUT_LOG = ".ipynb_checkpoints/generated.txt"

def show_generation(prompt, file_path=OUTPUT_LOG, prompt_number=None):

    #Shows the prompt and generated text, and saves them to a file.
    result = generate(prompt)
    
    #Print
    block_header = f"PROMPT{f' #{prompt_number}' if prompt_number else ''}:"
    print(block_header)
    print(prompt.strip())
    print("\nGENERATION:")
    print(result.strip())
    print("===========================\n")

    # Save to file (append)
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(block_header + "\n")
        f.write(prompt.strip() + "\n\n")
        f.write("GENERATION:\n")
        f.write(result.strip() + "\n")
        f.write("===========================\n")

In [9]:
#Generate multiple prompts
prompts = [
    "Tell me about a brave warrior.",
    "Describe a brave knight in medieval times.",
    "What are the challenges of medieval warfare?"
]

for i, p in enumerate(prompts, start=1):
    show_generation(p, prompt_number=i)

PROMPT #1:
Tell me about a brave warrior.

GENERATION:
Tell me about a brave warrior.  In the epic tale of "Beowulf," Beowulf is often considered one such brave warrior. Born in Sweden, he travels to Denmark at the request of King Hrothgar and his Danes to fight a monster named Grendel who has been terrorizing them for years.

Beowulf arrives with only sixteen men, but they quickly prove their mettle when facing off against the monster's strength and ferocity. During several battles, Beowulf demonstrates immense courage, intelligence, and physical prowess as the hero of this story. His bravery extends beyond physical combat; he also shows compassion towards those in need and always chooses actions driven by

PROMPT #2:
Describe a brave knight in medieval times.

GENERATION:
Describe a brave knight in medieval times.  In the annals of history, there are many tales of bravery and chivalry that speak to the spirit of the Middle Ages. One such tale revolves around Sir Lancelot of Camelot -